#Initialization

In [0]:
%sql
USE CATALOG workspace

# Read Bronze table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df AS
SELECT * FROM workspace.bronze.erp_cust_az12

#Silver Transformations

##Trimming

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_trimmed AS
SELECT TRIM(cid) AS cid,
       bdate,
       TRIM(gen) AS gen
FROM df

##Customer ID Cleanup

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_cid AS
SELECT CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LENGTH(cid))
            ELSE cid END AS cid,
       bdate, gen
FROM df_trimmed

##Birthdate Validation

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_bdate AS
SELECT cid,
       CASE WHEN bdate > CURRENT_DATE() THEN NULL
            ELSE bdate END AS bdate,
       gen
FROM df_cid

##Gender Normalization

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_gender AS
SELECT cid, bdate,
       CASE WHEN UPPER(gen) IN ('F', 'FEMALE') THEN 'Female'
            WHEN UPPER(gen) IN ('M', 'MALE')   THEN 'Male'
            ELSE 'n/a' END AS gen
FROM df_bdate

## Renaming Columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_renamed AS
SELECT cid   AS customer_number,
       bdate AS birth_date,
       gen   AS gender
FROM df_gender

## Sanity checks of dataframe

In [0]:
%sql
SELECT * FROM df_renamed LIMIT 10

#Writing Silver Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.erp_customers USING DELTA AS
SELECT * FROM df_renamed

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10